# DocAudit Agent — Evaluación (Corpus_GAJJIA)

Este notebook adapta lo mejor del notebook del compañero (métricas: F1/latencia/RAM) pero ejecutando **nuestro pipeline** (extractor/normalizador/validador/auditor) y **sin requerir PostgreSQL**.

Requisitos:
- Ollama corriendo (http://localhost:11434)
- Modelos: `llama3.2:3b` (texto), `qwen2.5vl:7b` (visión si el PDF es escaneado)
- `psutil` instalado (ya viene en requirements del proyecto)


In [ ]:
import os
import sys
import json
import time
import threading
from pathlib import Path
from typing import Any

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import psutil

from core.document_loader import extract_text_from_pdf_bytes, extract_text_from_scanned_pdf_bytes
from core.schema_models import DocSchema, SchemaField
from agents.extractor import extract_from_text
from core.normalizer import normalize_extracted
from core.validator import validate_extracted
from agents.auditor import audit_document

print("OK: imports")

In [ ]:
CORPUS_DIR = Path(r"c:\Users\gusta\Desktop\Maestria\0. TFM\DocAudit Agent\Corpus_GAJJIA\Corpus_GAJJIA")
DISABLE_RAG = True
USE_VISION_IF_NO_TEXT = True

pdf_paths = sorted(CORPUS_DIR.glob("*.pdf"))
gt_paths = sorted(CORPUS_DIR.glob("*.json"))
print("PDFs:", len(pdf_paths))
print("GT JSONs:", len(gt_paths))
print("Ejemplo PDFs:", [p.name for p in pdf_paths[:5]])
print("Ejemplo GT:", [p.name for p in gt_paths[:5]])

In [ ]:
if DISABLE_RAG:
    import core.rag as rag

    def _no_rag(queries: list[str], chunks: list[Any], top_k: int = 1, doc_id: str | None = None):
        return [[] for _ in queries]

    rag.retrieve_best_evidence_batch = _no_rag
    print("RAG desactivado para evaluación (más rápido)")

In [ ]:
def _guess_type(value: Any) -> str:
    if isinstance(value, bool):
        return "boolean"
    if isinstance(value, int) and not isinstance(value, bool):
        return "integer"
    if isinstance(value, float):
        return "number"
    if isinstance(value, str) and len(value) >= 10 and value[4:5] == "-" and value[7:8] == "-":
        return "date"
    return "string"


def build_schema_from_ground_truth(gt: dict[str, Any], *, name: str = "corpus_gajjia") -> DocSchema:
    fields: list[SchemaField] = []
    for k, v in gt.items():
        if k == "id_documento":
            continue
        fields.append(SchemaField(name=str(k), type=_guess_type(v), required=False, description=str(k)))
    return DocSchema(name=name, version="1.0", fields=fields)


def normalize_for_match(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return str(value).strip().lower()
    return str(value).strip().lower()


def field_level_f1(y_true: dict[str, Any], y_pred: dict[str, Any]) -> dict[str, Any]:
    keys = set(y_true.keys()) | set(y_pred.keys())
    tp = fp = fn = 0
    details: list[str] = []
    for k in sorted(keys):
        real = normalize_for_match(y_true.get(k))
        pred = normalize_for_match(y_pred.get(k))
        if real and pred and real == pred:
            tp += 1
            details.append(f"OK  {k}: '{pred}'")
        elif pred and real != pred:
            fp += 1
            details.append(f"FP  {k}: real='{real}' pred='{pred}'")
        elif real and not pred:
            fn += 1
            details.append(f"FN  {k}: real='{real}' pred=''")

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    exact = tp / len(keys) if keys else 0.0
    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "exact_match_rate": exact,
        "details": details,
    }


def run_with_ram_peak(fn):
    proc = psutil.Process(os.getpid())
    peak = 0
    running = True

    def monitor():
        nonlocal peak, running
        while running:
            rss = proc.memory_info().rss
            if rss > peak:
                peak = rss
            time.sleep(0.05)

    t = threading.Thread(target=monitor, daemon=True)
    t.start()
    try:
        start = time.perf_counter()
        out = fn()
        elapsed = time.perf_counter() - start
        return out, elapsed, peak
    finally:
        running = False
        t.join(timeout=1.0)


In [ ]:
def load_ground_truth_for_pdf(pdf_path: Path) -> dict[str, Any] | None:
    gt = pdf_path.with_suffix(".json")
    if gt.exists():
        return json.loads(gt.read_text(encoding="utf-8"))
    return None


def process_one(pdf_path: Path) -> dict[str, Any]:
    pdf_bytes = pdf_path.read_bytes()
    extracted = extract_text_from_pdf_bytes(pdf_bytes)
    pages = extracted.get("page_texts")
    text = (extracted.get("text") or "").strip()
    if USE_VISION_IF_NO_TEXT and not text:
        extracted_v = extract_text_from_scanned_pdf_bytes(pdf_bytes)
        pages = extracted_v.get("page_texts")
        text = (extracted_v.get("text") or "").strip()

    gt = load_ground_truth_for_pdf(pdf_path)
    if not gt:
        return {
            "file": pdf_path.name,
            "error": "No se encontró ground truth .json con el mismo nombre",
        }

    schema = build_schema_from_ground_truth(gt, name=f"gt_{pdf_path.stem}")

    def _run():
        raw = extract_from_text(text, schema, pages=pages if isinstance(pages, list) else None, doc_id=None)
        if isinstance(raw, dict) and "fields" in raw:
            fields = raw.get("fields") or {}
            details = raw.get("details") or {}
        else:
            fields = raw
            details = {}
        norm = normalize_extracted(fields, schema)
        val = validate_extracted(norm["normalized"], schema)
        rep = audit_document(schema, norm["normalized"], val, field_details=details)
        return {
            "schema": {"name": schema.name, "version": schema.version},
            "extracted": norm["normalized"],
            "validation": val,
            "report": rep,
        }

    result, seconds, peak_bytes = run_with_ram_peak(_run)
    metrics = field_level_f1(
        {k: v for k, v in gt.items() if k != "id_documento"},
        result.get("extracted") or {},
    )
    return {
        "file": pdf_path.name,
        "seconds": round(seconds, 3),
        "ram_peak_gb": round(peak_bytes / (1024**3), 3),
        "f1": round(metrics["f1"], 3),
        "precision": round(metrics["precision"], 3),
        "recall": round(metrics["recall"], 3),
        "exact_match_rate": round(metrics["exact_match_rate"], 3),
        "metrics_details": metrics["details"],
        "result": result,
    }


if not pdf_paths:
    print("No se encontraron PDFs en la carpeta.")
else:
    out = process_one(pdf_paths[0])
    print(json.dumps({k: v for k, v in out.items() if k not in {"result", "metrics_details"}}, ensure_ascii=False, indent=2))


In [ ]:
results: list[dict[str, Any]] = []
for p in pdf_paths:
    print("\n===", p.name, "===")
    r = process_one(p)
    results.append(r)
    if r.get("error"):
        print("ERROR:", r["error"])
        continue
    print(f"latencia_s={r['seconds']}  ram_peak_gb={r['ram_peak_gb']}  f1={r['f1']}  exact={r['exact_match_rate']}")

ok = [r for r in results if not r.get("error")]
if ok:
    avg_f1 = sum(r["f1"] for r in ok) / len(ok)
    avg_s = sum(r["seconds"] for r in ok) / len(ok)
    avg_ram = sum(r["ram_peak_gb"] for r in ok) / len(ok)
    print("\n--- RESUMEN ---")
    print("docs:", len(ok))
    print("f1_avg:", round(avg_f1, 3))
    print("latencia_avg_s:", round(avg_s, 3))
    print("ram_peak_avg_gb:", round(avg_ram, 3))
